Corrective Rag

In corrective Rag we use three categories appraoch

* MOST RELEVANT: Having atleast one document with relevancy score greater than upper threshold

* LEAST RELEVANT: Having no document relevancy score greater than lower threshold

* AMBIGOUS: Having all the documents relevancy less then upper theshold but atleast one document with relevancy score greater than lower theshold

In first one we do only contexual generation

in the second category we go for only web search

In the ambigous we go for both web and contexual appraoch

In [2]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import List, TypedDict
from langchain_core.messages import HumanMessage
from langchain_core.documents import Document

from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
import os 
import numpy as np
os.environ["HF_HOME"] = "E:\\huggingface_embedding_cache"
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True},
)



C:\Users\ndm60\AppData\Local\Temp\ipykernel_16616\1595770610.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1913.64it/s]


In [3]:
class State(TypedDict):
    question: str
    documents: list
    web_results: list
    generation: str

In [7]:
from langchain_chroma import Chroma
# ==========================================================
# CREATE VECTOR DATABASE
# ==========================================================
loader = TextLoader(
    "data.txt",
    encoding="utf-8"
)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

persist_directory='./vectorStore'



vectorStore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="my_docs",
    persist_directory=persist_directory,
)

retriever = vectorStore.as_retriever(
    search_kwargs={"k":3}
)

In [10]:
results = vectorStore.similarity_search(query="what is this topic mainly about", k=2)
print(results)

[Document(id='3b92b951-9278-449d-8474-04db6282e4e3', metadata={'source': 'data.txt'}, page_content='of products and services across artificial intelligence, cloud computing, advertising, mobile operating systems, productivity software, hardware, and more.'), Document(id='2922ff31-564f-453b-9fb3-66a90b14bd5a', metadata={'source': 'data.txt'}, page_content='# Google: A Comprehensive Overview')]
